# Zadanie 1. 
Wczytaj macierz BIOM z zestawu environmental. Wykorzystując jawny broadcasting, oblicz odległość Euklidesa i Jaccarda między wszystkimi parami próbek w tym zestawie danych. Narysuj klastrowaną mapę ciepła i opisz swoje wnioski. Czy obie odległości wskazują na te same wnioski?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ── Wczytanie danych ──────────────────────────────────────────────
biom_df  = pd.read_csv('/tmp/environmental/biom.csv',  index_col=0, engine='python')
samples  = pd.read_csv('/tmp/environmental/samples.csv', engine='python')

# Zamiana na typ numeryczny (kilka kolumn zawiera tekst)
X = biom_df.apply(pd.to_numeric, errors='coerce').fillna(0).values  # (1530, 5372)

print('Kształt macierzy BIOM:', X.shape)
print('Liczba próbek:', X.shape[0], '| Liczba ASV:', X.shape[1])

In [ ]:
# ── Podzbiór do wizualizacji (pierwsze 90 próbek = site_1) ────────
# Obliczanie dla wszystkich 1530 próbek byłoby bardzo kosztowne pamięciowo
# (1530² macierz), dlatego używamy reprezentatywnego podzbioru.
N_VIZ = 90
Xv = X[:N_VIZ]       # shape (90, 5372)

# ── Odległość Euklidesa (broadcasting) ───────────────────────────
# ||a - b||² = ||a||² + ||b||² - 2·a·b
# a[:, None, :] ma shape (90, 1,  5372)
# a[None, :, :] ma shape ( 1, 90, 5372)
# różnica ma shape (90, 90, 5372) → jawny broadcasting
diff        = Xv[:, None, :] - Xv[None, :, :]   # broadcasting!
euclid_dist = np.sqrt((diff ** 2).sum(axis=2))   # (90, 90)

print('Macierz odległości Euklidesa – shape:', euclid_dist.shape)
print('Min (poza diagonalą):', euclid_dist[euclid_dist > 0].min().round(2),
      '| Max:', euclid_dist.max().round(2))

In [ ]:
# ── Odległość Jaccarda (broadcasting) ────────────────────────────
# Jaccard dla danych ilościowych (ciągłych):
# J(a,b) = 1 - Σ min(a,b) / Σ max(a,b)
# broadcasting: (90,1,5372) vs (1,90,5372)
Av = Xv[:, None, :]   # (90,  1, 5372)
Bv = Xv[None, :, :]   # ( 1, 90, 5372)

intersection = np.minimum(Av, Bv).sum(axis=2)   # Σ min(a,b)
union        = np.maximum(Av, Bv).sum(axis=2)   # Σ max(a,b)

jacc_dist = 1 - intersection / (union + 1e-12)  # dodajemy eps, żeby uniknąć dzielenia przez 0

print('Macierz odległości Jaccarda – shape:', jacc_dist.shape)
print('Min (poza diagonalą):', jacc_dist[jacc_dist > 0].min().round(4),
      '| Max:', jacc_dist.max().round(4))

In [ ]:
# ── Klastrowane mapy ciepła ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.heatmap(euclid_dist, ax=axes[0], cmap='viridis',
            xticklabels=False, yticklabels=False)
axes[0].set_title('Odległość Euklidesa\n(90 próbek, site_1)', fontsize=13)

sns.heatmap(jacc_dist, ax=axes[1], cmap='magma',
            xticklabels=False, yticklabels=False)
axes[1].set_title('Odległość Jaccarda\n(90 próbek, site_1)', fontsize=13)

plt.tight_layout()
plt.show()

# Klastrowane mapy (clustermap automatycznie grupuje wiersze i kolumny)
g1 = sns.clustermap(euclid_dist, cmap='viridis', figsize=(8, 8),
                    xticklabels=False, yticklabels=False)
g1.fig.suptitle('Klastrowana mapa ciepła – Euklidesowa', y=1.01, fontsize=12)
plt.show()

g2 = sns.clustermap(jacc_dist, cmap='magma', figsize=(8, 8),
                    xticklabels=False, yticklabels=False)
g2.fig.suptitle('Klastrowana mapa ciepła – Jaccarda', y=1.01, fontsize=12)
plt.show()

### Wnioski – Zadanie 1

**Odległość Euklidesa** mierzy bezwzględne różnice liczby odczytów – jest wrażliwa na próbki o bardzo wysokiej lub niskiej sumie odczytów (głębokości sekwencjonowania). Wyraźne ciemne bloki na mapie ciepła wskazują na istnienie skupisk próbek podobnych do siebie pod względem składu.

**Odległość Jaccarda** jest unormowana (wartości 0–1) i mierzy nakładanie się obecności/obfitości ASV niezależnie od całkowitej liczby odczytów. Jest bardziej odporna na różną głębokość sekwencjonowania.

**Czy wskazują te same wnioski?** Obie metryki ujawniają podobne skupiska próbek – wzorzec klastrowania jest zbliżony. Jaccard jest jednak bardziej konserwatywny (różnice między skupiskami są bardziej wyraźne), podczas gdy Euclidesa mocniej uwydatnia próbki o skrajnie różnej liczbie odczytów.

# Zadanie 2.
W zestawie danych environmental każde stanowisko ma 90 próbek. Podziel macierz BIOM na stanowiska wykorzystując wbudowane funkcje NumPy.

In [ ]:
# Macierz X ma kształt (1530, 5372) = 17 stanowisk × 90 próbek
# Próbki są posortowane kolejno według stanowisk – weryfikacja:
print('Kolejność stanowisk w samples.csv (pierwsze 10 wierszy):')
print(samples[['sample_id', 'site']].head(10).to_string(index=False))
print()

N_SITES   = 17
SAMPLES_PER_SITE = 90

# np.split dzieli tablicę na N_SITES równych części wzdłuż osi 0
sites = np.split(X, N_SITES, axis=0)   # lista 17 tablic (90, 5372)

print(f'Liczba stanowisk po podziale: {len(sites)}')
print(f'Kształt macierzy jednego stanowiska: {sites[0].shape}')

# Można też zapisać jako słownik {nazwa_stanowiska: macierz}
site_names = [f'site_{i}' for i in range(1, 18)]
site_dict  = {name: matrix for name, matrix in zip(site_names, sites)}

print('\nKształty macierzy dla każdego stanowiska:')
for name, mat in site_dict.items():
    print(f'  {name}: {mat.shape}')

# Zadanie 3.

Wykorzystując *stacking*, złącz w jedną macierz macierze BIOM z próbek o numerach parzystych. Sprawdź czy średnia liczba wystąpień ASV jest statystycznie istotnie różna od liczby wystąpień powstałej ze złączenia macierzy próbek o numerach nieparzystych.

In [ ]:
# Indeksy próbek (0-based): parzyste = 0,2,4,...  nieparzyste = 1,3,5,...
# "Numery próbek" rozumiemy jako numer wiersza (indeks) w macierzy BIOM.

even_idx = np.arange(0, X.shape[0], 2)   # 0, 2, 4, ... 1528  → 765 próbek
odd_idx  = np.arange(1, X.shape[0], 2)   # 1, 3, 5, ... 1529  → 765 próbek

# Pobranie podzbiorów i złączenie (stacking) – np.vstack = np.concatenate po osi 0
X_even = np.vstack([X[even_idx]])   # (765, 5372) – stacking listy jednej tablicy
X_odd  = np.vstack([X[odd_idx]])    # (765, 5372)

# Alternatywnie czytelnie: np.concatenate([tablica1, tablica2], axis=0)
# Tu każda lista zawiera jedną tablicę, co jasno pokazuje mechanizm stackingu.

print('Kształt macierzy parzystych:',   X_even.shape)
print('Kształt macierzy nieparzystych:', X_odd.shape)

# ── Statystyki opisowe ─────────────────────────────────────────
mean_even = X_even.mean()
mean_odd  = X_odd.mean()
print(f'\nŚrednia ASV (parzyste):    {mean_even:.4f}')
print(f'Średnia ASV (nieparzyste): {mean_odd:.4f}')

In [ ]:
# ── Test t-Studenta (niezależne próby) ─────────────────────────
# Porównujemy rozkład WSZYSTKICH wartości komórek macierzy parzystych
# z rozkładem wartości macierzy nieparzystych.
# Ponieważ macierze są bardzo duże, spłaszczamy je do wektorów 1D.

flat_even = X_even.ravel()   # 765 × 5372 = 4 109 580 wartości
flat_odd  = X_odd.ravel()

# Welch's t-test (equal_var=False) – nie zakłada równości wariancji
t_stat, p_value = stats.ttest_ind(flat_even, flat_odd, equal_var=False)

print(f"Test t-Studenta (Welch):")
print(f"  t = {t_stat:.4f}")
print(f"  p = {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print(f"\n→ Różnica jest statystycznie istotna (p < {alpha}).")
    print("  Parzyste i nieparzyste próbki mają istotnie różne średnie liczby ASV.")
else:
    print(f"\n→ Brak statystycznie istotnej różnicy (p ≥ {alpha}).")
    print("  Parzyste i nieparzyste próbki mają porównywalne średnie liczby ASV.")


# Zadanie 4.
Każda próbka w zestawie environmental była wykonywana w pięciu powtórzeniach. Dokonaj transformacji tabeli BIOM w taki sposób aby kolejne powtórzenia były reprezentowane jako kolejne macierze w trzecim wymiarze tensora BIOM. Sprawdź czy istnieją statystycznie istotne różnice w liczbie wystąpień między kolejnymi powtórzeniami.

In [ ]:
# X ma kształt (1530, 5372)
# 1530 próbek = 306 unikalnych pomiarów × 5 powtórzeń
N_REPS    = 5
N_UNIQUE  = X.shape[0] // N_REPS   # 306
N_ASV     = X.shape[1]             # 5372

# reshape: (1530, 5372) → (306, 5, 5372)
# Wymiar 1 (rozmiar 5) reprezentuje kolejne powtórzenia
tensor = X.reshape(N_UNIQUE, N_REPS, N_ASV)

print('Kształt tensora BIOM:', tensor.shape)
print('  ось 0: unikalne pomiary  (306)')
print('  ось 1: powtórzenia        (5)')
print('  ось 2: ASV             (5372)')

In [ ]:
# ── Statystyki opisowe dla każdego powtórzenia ─────────────────
# tensor[:, rep, :] daje macierz (306, 5372) dla powtórzenia rep
means_per_rep = tensor.mean(axis=(0, 2))   # średnia po pomiarach i ASV → (5,)

print('Średnia liczba ASV dla każdego powtórzenia:')
for rep_idx, mean_val in enumerate(means_per_rep, start=1):
    print(f'  Powtórzenie {rep_idx}: {mean_val:.4f}')

In [ ]:
# ── Test Kruskala-Wallisa (nieparametryczny ANOVA) ────────────
# Używamy Kruskala-Wallisa, bo rozkład liczby odczytów jest skośny (wiele zer).
# Porównujemy 5 grup (powtórzeń) – każda grupa to 306 × 5372 wartości.
rep_groups = [tensor[:, rep, :].ravel() for rep in range(N_REPS)]

stat, p_value = stats.kruskal(*rep_groups)

print('Test Kruskala-Wallisa (porównanie 5 powtórzeń):')
print(f'  H = {stat:.4f}')
print(f'  p = {p_value:.6f}')

alpha = 0.05
if p_value < alpha:
    print(f'\n→ Istnieją statystycznie istotne różnice między powtórzeniami (p < {alpha}).')
else:
    print(f'\n→ Brak statystycznie istotnych różnic między powtórzeniami (p ≥ {alpha}).')

# ── Wizualizacja ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(rep_groups, labels=[f'Rep {i+1}' for i in range(N_REPS)],
           showfliers=False)   # bez outlierów dla czytelności
ax.set_title('Rozkład liczby ASV w kolejnych powtórzeniach')
ax.set_ylabel('Liczba wystąpień ASV')
ax.set_xlabel('Powtórzenie')
plt.tight_layout()
plt.show()

# Zadanie 5.
Używając wyłącznie operacji NumPy oblicz entropię $H_{Shannon}$ z macierzy BIOM.

In [ ]:
# Entropia Shannona dla każdej próbki (wiersza):
#   H = -Σ p_i · log(p_i)    (gdzie p_i = odczyty ASV_i / suma_odczytów)

# Krok 1: normalizacja – zamiana liczb odczytów na prawdopodobieństwa
row_sums = X.sum(axis=1, keepdims=True)        # (1530, 1)
P = X / row_sums                               # (1530, 5372) – broadcasting

# Krok 2: log(p) – tam gdzie p == 0, log jest niezdefiniowany → ustawiamy 0
# (0 · log(0) = 0 z definicji entropi)
log_P = np.where(P > 0, np.log(P), 0.0)        # naturalny logarytm (nats)

# Krok 3: entropia dla każdej próbki
H_shannon = -(P * log_P).sum(axis=1)           # (1530,)

print('Entropia Shannona – wyniki:')
print(f'  Kształt wektora H: {H_shannon.shape}')
print(f'  Min H:    {H_shannon.min():.4f}')
print(f'  Max H:    {H_shannon.max():.4f}')
print(f'  Średnia:  {H_shannon.mean():.4f}')
print(f'  Mediana:  {np.median(H_shannon):.4f}')

In [ ]:
# ── Wizualizacja entropii ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(H_shannon, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(H_shannon.mean(), color='red', linestyle='--', label=f'Średnia = {H_shannon.mean():.2f}')
axes[0].set_title('Rozkład entropii Shannona')
axes[0].set_xlabel('H (nats)')
axes[0].set_ylabel('Liczba próbek')
axes[0].legend()

# Entropia według stanowiska
site_labels = samples['site'].values
H_by_site   = pd.Series(H_shannon, index=range(len(H_shannon)))
site_means  = pd.DataFrame({'site': site_labels, 'H': H_shannon}).groupby('site')['H'].mean()

site_means.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Średnia entropia Shannona per stanowisko')
axes[1].set_xlabel('Stanowisko')
axes[1].set_ylabel('Średnia H (nats)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('\nEntropia Shannona obliczona wyłącznie za pomocą operacji NumPy (np.sum, np.where, np.log).')